# **Flow 3: Persiapan Data & Arsitektur Model GNN-ODE**

## **3.1 Setup & Load Data**

**Teknik Code:** Mount Drive, `pip install torch_geometric torchdiffeq, pickle.load(), read_parquet()`

**Cara Kerja:** Menyiapkan environment PyTorch Geometric (library GNN) dan torchdiffeq (library ODE solver), lalu memuat kembali graf dan data speed yang sudah disimpan di Flow 2.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/GNN_ODE_Traffic')

import torch
print(f"Torch version: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

!pip install torch_geometric -q
!pip install torchdiffeq -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Torch version: 2.11.0+cpu, CUDA: False
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.5 MB/s eta 0:00:00


In [ ]:
import pickle
import pandas as pd
import numpy as np

with open('sudirman_thamrin_graph_final.pkl', 'rb') as f:
    G = pickle.load(f)

df_speed = pd.read_parquet('synthetic_traffic_speed.parquet')

with open('metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"df_speed: {df_speed.shape}")

Graph: 2669 nodes, 5783 edges
df_speed: (11658528, 9)


## 3.2 Agregasi Speed Edge ke Node

**Teknik Code:** `pd.concat() + groupby().mean() + pivot()`

**Cara Kerja:** Karena data kecepatan awalnya per-edge (ruas jalan) sementara GNN bekerja per-node, kode ini merata-ratakan kecepatan semua edge yang menyentuh sebuah node untuk menghasilkan satu nilai kecepatan per node per waktu, membentuk matriks [waktu x node].



In [ ]:
# Map speed edge ke node: rata-rata semua edge yang terhubung ke tiap node
node_list = list(G.nodes())
node_id_to_idx = {node_id: i for i, node_id in enumerate(node_list)}
num_nodes = len(node_list)

# Agregasi: untuk tiap (node, timestamp), rata-ratakan speed dari edge yang terhubung
df_speed['u_idx'] = df_speed['u'].map(node_id_to_idx)
df_speed['v_idx'] = df_speed['v'].map(node_id_to_idx)

# Gabungkan kontribusi dari sisi u dan v (edge menyentuh 2 node)
df_u = df_speed[['u_idx', 'timestamp', 'speed_kph']].rename(columns={'u_idx': 'node_idx'})
df_v = df_speed[['v_idx', 'timestamp', 'speed_kph']].rename(columns={'v_idx': 'node_idx'})
df_node_speed = pd.concat([df_u, df_v], ignore_index=True)

node_speed_agg = df_node_speed.groupby(['node_idx', 'timestamp'])['speed_kph'].mean().reset_index()

# Pivot jadi matrix [num_timestamps, num_nodes]
speed_matrix = node_speed_agg.pivot(index='timestamp', columns='node_idx', values='speed_kph')
speed_matrix = speed_matrix.reindex(columns=range(num_nodes))  # pastikan semua node ada kolomnya
speed_matrix = speed_matrix.fillna(speed_matrix.mean())  # isolated node (jarang) -> isi mean

print(f"Speed matrix shape: {speed_matrix.shape}")  # (num_timestamps, num_nodes)
speed_matrix.head()

Speed matrix shape: (2016, 2669)


node_idx,0,1,2,3,4,5,6,7,8,9,...,2659,2660,2661,2662,2663,2664,2665,2666,2667,2668
timestamp,,,,,,,,,,,,,,,,,,,,,
2026-01-06 00:00:00,48.521701,47.409102,39.926685,45.271206,36.389018,49.026781,35.573796,38.466693,48.056142,43.324128,...,25.747859,29.306481,27.786358,46.269574,35.810662,36.295594,30.318455,35.760130,49.984235,39.764818
2026-01-06 00:05:00,46.947211,45.951423,39.653355,43.767089,36.564278,48.725105,34.879111,37.762712,48.397469,42.979540,...,25.178863,28.833903,27.809885,46.514418,36.627653,35.890427,27.908720,36.899493,50.811587,39.106789
2026-01-06 00:10:00,47.781076,47.244295,39.899315,44.471765,37.169706,48.797084,36.298029,37.967574,48.884072,42.923898,...,25.502372,29.181605,28.290300,46.087084,36.526335,35.299671,28.701959,37.511543,49.961612,40.187479
2026-01-06 00:15:00,46.717504,45.987549,39.950184,44.023699,35.959029,48.954650,35.051075,39.063628,48.511030,43.573796,...,26.222141,29.255202,27.453776,46.060290,36.628673,35.331437,27.511997,35.749855,49.002066,39.539139
2026-01-06 00:20:00,47.800567,46.437206,39.501444,44.302832,36.187150,48.668712,35.011735,37.822612,47.319662,45.077944,...,25.354651,29.349088,27.674790,46.187323,37.133533,37.022427,27.643469,35.110225,49.058212,39.002568


## **3.3 Konstruksi Edge Index untuk Graph Neural Network**

**Teknik Code:** `torch_geometric.data.`Data format, konversi edge list ke `torch.tensor`

**Cara Kerja:** Mengubah daftar edge graf (pasangan node) menjadi format edge_index standar PyTorch Geometric berbentuk [2, jumlah_edge], yang menjadi struktur konektivitas graf yang dipakai model GNN.

In [ ]:
from torch_geometric.data import Data

edge_index_list = []
for u, v, k in G.edges(keys=True):
    edge_index_list.append([node_id_to_idx[u], node_id_to_idx[v]])

edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
print(f"edge_index shape: {edge_index.shape}")  # [2, num_edges]

edge_index shape: torch.Size([2, 5783])


## **3.4 Pembuatan Dataset Time-Series**

**Teknik Code:** Custom `torch.utils.data.Dataset` (sliding window `seq_len` & `horizon`)

**Cara Kerja:** Membungkus matriks kecepatan menjadi pasangan input-output: input berupa 12 langkah waktu historis (1 jam), output berupa nilai kecepatan pada langkah waktu target (horizon) yang ingin diprediksi.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TrafficDataset(Dataset):
    def __init__(self, speed_matrix, seq_len=12, horizon=1):
        """
        speed_matrix: numpy array [num_timestamps, num_nodes]
        seq_len: jumlah step historis dipakai sbg input (12 x 5menit = 1 jam)
        horizon: berapa step ke depan yang diprediksi (1 x 5menit = 5 menit ke depan)
        """
        self.data = speed_matrix.values.astype(np.float32)
        self.seq_len = seq_len
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.seq_len - self.horizon + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]              # [seq_len, num_nodes]
        y = self.data[idx + self.seq_len + self.horizon - 1]  # [num_nodes]
        return torch.tensor(x), torch.tensor(y)

dataset = TrafficDataset(speed_matrix, seq_len=12, horizon=1)
print(f"Total samples: {len(dataset)}")

x_sample, y_sample = dataset[0]
print(f"x shape: {x_sample.shape}, y shape: {y_sample.shape}")

Total samples: 2004
x shape: torch.Size([12, 2669]), y shape: torch.Size([2669])


## **3.5 Definisi Arsitektur GNN-ODE**

**Teknik Code:** `GCNConv` (Graph Convolution) dibungkus dalam `ODEFunc`, diintegrasikan via `torchdiffeq.odeint()`

**Cara Kerja:** Model terdiri dari encoder (Linear), fungsi turunan ODEFunc yang berisi 2 layer GCN (menangkap struktur spasial graf), lalu status tersembunyi diintegrasikan secara kontinu dari waktu t=0 ke t=target menggunakan solver ODE (rk4), dan didekode kembali menjadi prediksi kecepatan lewat decoder (Linear).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torchdiffeq import odeint

class ODEFunc(nn.Module):
    """f_theta(h, t) = GCN(h) -- ini yang diintegrasikan odeint"""
    def __init__(self, hidden_dim, edge_index):
        super().__init__()
        self.edge_index = edge_index
        self.gcn1 = GCNConv(hidden_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, t, h):
        # h: [num_nodes, hidden_dim]
        out = F.relu(self.gcn1(h, self.edge_index))
        out = self.gcn2(out, self.edge_index)
        return out

class GNN_ODE(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_index, solver='rk4'):
        super().__init__()
        self.encoder = nn.Linear(in_dim, hidden_dim)
        self.odefunc = ODEFunc(hidden_dim, edge_index)
        self.decoder = nn.Linear(hidden_dim, 1)  # output: 1 nilai speed per node
        self.solver = solver

    def forward(self, x, t_span):
        """
        x: [batch, seq_len, num_nodes] -> ambil snapshot terakhir sbg initial state
        t_span: tensor waktu integrasi, misal torch.tensor([0., 1.])
        """
        h0 = x[:, -1, :].unsqueeze(-1)          # [batch, num_nodes, 1] -- state awal = snapshot terakhir
        batch_size = h0.shape[0]

        h0 = self.encoder(h0)                    # [batch, num_nodes, hidden_dim]

        outputs = []
        for b in range(batch_size):
            h_traj = odeint(self.odefunc, h0[b], t_span, method=self.solver)  # [len(t_span), num_nodes, hidden_dim]
            h_final = h_traj[-1]                 # ambil state di t akhir
            outputs.append(h_final)
        h_final_batch = torch.stack(outputs)     # [batch, num_nodes, hidden_dim]

        pred = self.decoder(h_final_batch).squeeze(-1)  # [batch, num_nodes]
        return pred

## **3.6 Uji Coba Forward Pass Model**

**Teknik Code:** Instansiasi model + 1 batch dari `DataLoader` + cek `torch.isnan()`

**Cara Kerja:** Menjalankan satu batch data ke model untuk memastikan bentuk output sesuai ekspektasi, tidak ada nilai NaN, dan loss (MSE) bisa dihitung — sebagai sanity check sebelum training penuh.

In [ ]:
model = GNN_ODE(in_dim=1, hidden_dim=32, edge_index=edge_index, solver='rk4')

loader = DataLoader(dataset, batch_size=4, shuffle=True)
x_batch, y_batch = next(iter(loader))
print(f"x_batch: {x_batch.shape}, y_batch: {y_batch.shape}")

t_span = torch.tensor([0., 1.])  # integrasi dari t=0 ke t=1 (representasi 1 step horizon)

pred = model(x_batch, t_span)
print(f"Prediksi shape: {pred.shape}")  # harus [batch, num_nodes] sama seperti y_batch

# Cek gak ada NaN
print(f"Ada NaN? {torch.isnan(pred).any().item()}")

# Cek loss dasar bisa dihitung
loss_fn = nn.MSELoss()
loss = loss_fn(pred, y_batch)
print(f"Sample loss: {loss.item():.4f}")

x_batch: torch.Size([4, 12, 2669]), y_batch: torch.Size([4, 2669])
Prediksi shape: torch.Size([4, 2669])
Ada NaN? False
Sample loss: 209.3435


## **3.7 Penyimpanan Konfigurasi**

**Teknik Code:** `pickle.dump()` (dict config) + `torch.save()` (edge_index) + `to_parquet()` (speed_matrix)

**Cara Kerja:** Menyimpan seluruh konfigurasi model (dimensi, seq_len, horizon, solver) beserta edge_index dan matriks kecepatan level-node agar flow training berikutnya tinggal load, tanpa re-transformasi data.

In [ ]:
import pickle

# Simpan konfigurasi model & dataset (bukan weights, karena belum training)
notebook3_config = {
    'in_dim': 1,
    'hidden_dim': 32,
    'seq_len': 12,
    'horizon': 1,
    'solver': 'rk4',
    'num_nodes': num_nodes,
    'edge_index_shape': list(edge_index.shape),
}

with open('notebook3_config.pkl', 'wb') as f:
    pickle.dump(notebook3_config, f)

# Simpan edge_index & speed_matrix biar Notebook 4 tinggal load, gak perlu ulang transform
torch.save(edge_index, 'edge_index.pt')
speed_matrix.to_parquet('speed_matrix_node_level.parquet')